In [1]:
import os, re, hashlib, zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations

# =========================
# CONFIG
# =========================
DATA_DIR = "."   # carpeta donde están tus 3 archivos
YEAR_FOCUS = 2025

BASE_XLSX   = os.path.join(DATA_DIR, "Base de Datos Scopus.xlsx")
SCIMAGO_XLSX= os.path.join(DATA_DIR, "scimagojr2024.xlsx")
SCOPUS_CSV  = os.path.join(DATA_DIR, "scopus_export_Dec 30-2025_3b154878-6066-4ab2-a031-9beeab49cf0a.csv")

OUT_DIR = os.path.join(DATA_DIR, "outputs")
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

In [2]:
# =========================
# HELPERS
# =========================
def clean_issn_token(x: str) -> str:
    x = str(x).strip()
    x = re.sub(r"[^0-9Xx]", "", x)
    return x.upper()

def build_pub_id(row) -> str:
    eid = row.get("EID")
    doi = row.get("DOI")
    if pd.notna(eid) and str(eid).strip():
        return str(eid).strip()
    if pd.notna(doi) and str(doi).strip():
        return "doi:" + str(doi).strip().lower()
    key = f"{str(row.get('Title','')).strip().lower()}|{row.get('Year','')}|{str(row.get('Source title','')).strip().lower()}"
    return "h:" + hashlib.md5(key.encode("utf-8")).hexdigest()

def savefig(fname: str):
    path = os.path.join(FIG_DIR, fname)
    plt.tight_layout()
    plt.savefig(path, dpi=220, bbox_inches="tight")
    plt.close()
    return path

def summary_counts(df, qcol="quartile"):
    vc = df[qcol].value_counts().reindex(["Q1","Q2","Q3","Q4","SQ"]).fillna(0).astype(int)
    total = int(vc.sum())
    out = pd.DataFrame({"count": vc, "share": (vc/total).round(4)})
    out.loc["Total"] = {"count": total, "share": 1.0}
    return out

def split_author_ids(s):
    if pd.isna(s):
        return []
    return [x.strip() for x in str(s).split(";") if x.strip()]

In [3]:
# =========================
# LOAD
# =========================
base = pd.read_excel(BASE_XLSX)
sci  = pd.read_excel(SCIMAGO_XLSX)
sc   = pd.read_csv(SCOPUS_CSV)

# Normalize base
base["author_id"] = base["ID SCOPUS"].astype("string").str.strip()
base["ESCUELA"]   = base["ESCUELA"].astype("string").str.strip()
base["DOCENTE"]   = base["DOCENTE"].astype("string").str.strip()

# Normalize scopus
sc["Year"] = pd.to_numeric(sc["Year"], errors="coerce").astype("Int64")
sc["pub_id"] = sc.apply(build_pub_id, axis=1)

In [4]:
# =========================
# QUARTILES: ISSN map + title fallback
# =========================
sci["q"] = sci["SJR Best Quartile"].astype("string").str.strip().str.upper()
sci.loc[~sci["q"].isin(["Q1","Q2","Q3","Q4"]), "q"] = pd.NA

issn_map = {}
issn_lists = sci["Issn"].astype("string").fillna("").apply(
    lambda s: [clean_issn_token(t) for t in re.split(r"[;,]", str(s)) if clean_issn_token(t)]
)
for idx, lst in issn_lists.items():
    q = sci.loc[idx, "q"]
    if pd.isna(q):
        continue
    for issn in lst:
        if issn and issn not in issn_map:
            issn_map[issn] = q

sci["title_norm"] = sci["Title"].astype("string").str.lower().str.strip()
title_map = sci.dropna(subset=["q"]).set_index("title_norm")["q"].to_dict()

def assign_quartile(row):
    issn_raw = row.get("ISSN")
    if pd.notna(issn_raw) and str(issn_raw).strip():
        parts = re.split(r"[;,]", str(issn_raw))
        cands = [clean_issn_token(p) for p in parts if clean_issn_token(p)]
        for c in cands:
            q = issn_map.get(c)
            if q in ["Q1","Q2","Q3","Q4"]:
                return q, "issn"
    t = str(row.get("Source title","")).lower().strip()
    q = title_map.get(t)
    if q in ["Q1","Q2","Q3","Q4"]:
        return q, "title"
    return "SQ", "none"

sc[["quartile","quartile_match"]] = sc.apply(assign_quartile, axis=1, result_type="expand")

In [5]:
# =========================
# EXPLODE AUTHORS + JOIN UTB
# =========================
sc["author_id_list"] = sc["Author(s) ID"].apply(split_author_ids)
auth = sc.loc[sc.index.repeat(sc["author_id_list"].str.len())].copy()
auth["author_id"] = [aid for lst in sc["author_id_list"] for aid in lst] if sc["author_id_list"].str.len().sum() else []
auth = auth[auth["author_id"] != ""]

auth_utb = auth.merge(
    base[["author_id","DOCENTE","CODIGO","ESCUELA"]],
    on="author_id",
    how="inner"
)

# Three counting bases
pub_utb = (auth_utb.groupby("pub_id")
           .agg(
               Year=("Year", "first"),
               Title=("Title","first"),
               Source=("Source title","first"),
               ISSN=("ISSN","first"),
               DOI=("DOI","first"),
               EID=("EID","first"),
               quartile=("quartile","first"),
               escuelas=("ESCUELA", lambda s: sorted(set(s.dropna()))),
               docentes=("DOCENTE", lambda s: sorted(set(s.dropna()))),
           )
           .reset_index())

author_contrib = auth_utb.drop_duplicates(["author_id","pub_id"]).copy()
unit_part      = auth_utb.drop_duplicates(["ESCUELA","pub_id"]).copy()

In [6]:
# =========================
# TABLES USED BY FIGURES
# =========================
uni_auth = summary_counts(author_contrib[author_contrib["Year"]==YEAR_FOCUS])
unit_yr = unit_part[unit_part["Year"]==YEAR_FOCUS].copy()

unit_table = (unit_yr.pivot_table(index="ESCUELA", columns="quartile", values="pub_id",
                                 aggfunc="nunique", fill_value=0)
              .reindex(columns=["Q1","Q2","Q3","Q4","SQ"], fill_value=0))
unit_table["Total"] = unit_table.sum(axis=1)

In [7]:
# =========================
# EXPORTS: CSV + Excel (antes de las figuras)
# =========================
import os
import pandas as pd

OUT_DIR = os.path.join(DATA_DIR, "outputs")
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

def export_df(df, name):
    df.to_csv(os.path.join(OUT_DIR, f"{name}.csv"), index=True)

# --- Resúmenes 2025 (3 bases de conteo) ---
uni_unique = summary_counts(pub_utb[pub_utb["Year"]==YEAR_FOCUS])                       # papers únicos UTB
uni_auth   = summary_counts(author_contrib[author_contrib["Year"]==YEAR_FOCUS])        # autor–paper
uni_unit   = summary_counts(unit_part[unit_part["Year"]==YEAR_FOCUS])                  # escuela–paper

export_df(uni_unique, f"UNI_{YEAR_FOCUS}_unique_pubs")
export_df(uni_auth,   f"UNI_{YEAR_FOCUS}_authorship_contrib")
export_df(uni_unit,   f"UNI_{YEAR_FOCUS}_unit_participation")

# --- Universidad por año (papers únicos) ---
uni_year = (pub_utb.groupby(["Year","quartile"])["pub_id"]
            .nunique()
            .unstack(fill_value=0)
            .reindex(columns=["Q1","Q2","Q3","Q4","SQ"], fill_value=0))
uni_year["Total"] = uni_year.sum(axis=1)
export_df(uni_year, "UNI_by_year_unique_pubs")

# --- Comparativo por escuela (participación escuela–paper, YEAR_FOCUS) ---
unit_yr = unit_part[unit_part["Year"]==YEAR_FOCUS].copy()

unit_table = (unit_yr.pivot_table(index="ESCUELA", columns="quartile", values="pub_id",
                                  aggfunc="nunique", fill_value=0)
              .reindex(columns=["Q1","Q2","Q3","Q4","SQ"], fill_value=0))
unit_table["Total"] = unit_table.sum(axis=1)
unit_table["Q1+Q2"] = unit_table["Q1"] + unit_table["Q2"]

researchers_per_unit = base.groupby("ESCUELA")["author_id"].nunique().rename("n_researchers")
unit_table = unit_table.join(researchers_per_unit, how="left")
unit_table["Pubs_per_researcher"] = (unit_table["Total"]/unit_table["n_researchers"]).round(2)
unit_table = unit_table.sort_values("Total", ascending=False)

export_df(unit_table, f"Schools_{YEAR_FOCUS}_unit_participation")

# --- Top investigadores por escuela (authorship basis) ---
top_doc = (author_contrib[author_contrib["Year"]==YEAR_FOCUS]
           .groupby(["ESCUELA","DOCENTE"])["pub_id"].nunique()
           .rename("authorship_pubs")
           .reset_index()
           .sort_values(["ESCUELA","authorship_pubs"], ascending=[True, False]))
top_doc.to_csv(os.path.join(OUT_DIR, f"Top_researchers_{YEAR_FOCUS}_authorship.csv"), index=False)

# --- Matriz de colaboración entre escuelas (papers que conectan ≥2 escuelas) ---
from itertools import combinations
units = sorted(base["ESCUELA"].dropna().unique())
collab = pd.DataFrame(0, index=units, columns=units, dtype=int)

pub_year = pub_utb[pub_utb["Year"]==YEAR_FOCUS]
for _, r in pub_year.iterrows():
    escs = r["escuelas"]
    if not escs or len(escs) < 2:
        continue
    escs = [e for e in escs if e in collab.index]
    for a, b in combinations(sorted(set(escs)), 2):
        collab.loc[a, b] += 1
        collab.loc[b, a] += 1

export_df(collab, f"Collaboration_matrix_{YEAR_FOCUS}")

# --- SQ audit: fuentes con más SQ (papers únicos) ---
sq_sources = (pub_year[pub_year["quartile"]=="SQ"]["Source"]
              .value_counts()
              .rename_axis("Source title")
              .reset_index(name="SQ_unique_pubs"))
sq_sources.to_csv(os.path.join(OUT_DIR, f"SQ_sources_{YEAR_FOCUS}.csv"), index=False)

# --- Exporta también las tablas base “auditables” (opcional pero recomendado) ---
pub_utb.to_csv(os.path.join(OUT_DIR, f"UTB_publications_unique_{YEAR_FOCUS}.csv"), index=False)
author_contrib.to_csv(os.path.join(OUT_DIR, f"UTB_authorship_pairs_{YEAR_FOCUS}.csv"), index=False)
unit_part.to_csv(os.path.join(OUT_DIR, f"UTB_unit_participation_pairs_{YEAR_FOCUS}.csv"), index=False)

# --- Excel multi-hoja (todo en un solo archivo) ---
excel_path = os.path.join(OUT_DIR, f"scopus_metrics_{YEAR_FOCUS}.xlsx")
with pd.ExcelWriter(excel_path) as xw:
    uni_unique.to_excel(xw, sheet_name="UNI_unique_pubs")
    uni_auth.to_excel(xw, sheet_name="UNI_authorship")
    uni_unit.to_excel(xw, sheet_name="UNI_unit_participation")
    uni_year.to_excel(xw, sheet_name="UNI_by_year_unique")
    unit_table.to_excel(xw, sheet_name="Schools_participation")
    top_doc.to_excel(xw, sheet_name="Top_researchers", index=False)
    collab.to_excel(xw, sheet_name="Collaboration_matrix")
    sq_sources.to_excel(xw, sheet_name="SQ_sources", index=False)

print("✅ Exportes listos en:", OUT_DIR)
print("✅ Excel:", excel_path)

✅ Exportes listos en: ./outputs
✅ Excel: ./outputs/scopus_metrics_2025.xlsx


In [12]:
# =========================
# FIGURE ETD: Stacked bar by researcher (authorship basis)
# =========================
ETD_NAME = "ESCUELA DE TRANSFORMACIÓN DIGITAL"  # debe coincidir con tu Base
etd_ac = author_contrib[
    (author_contrib["Year"] == YEAR_FOCUS) &
    (author_contrib["ESCUELA"] == ETD_NAME)
].copy()

# Conteo por docente y cuartil (papers únicos por docente, en modo autor–paper)
etd_doc_q = (etd_ac.groupby(["DOCENTE","quartile"])["pub_id"].nunique()
             .unstack(fill_value=0)
             .reindex(columns=["Q1","Q2","Q3","Q4","SQ"], fill_value=0))
etd_doc_q["Total"] = etd_doc_q.sum(axis=1)

# Orden: de menor a mayor para que se vea “creciente” en la gráfica
etd_doc_q = etd_doc_q.sort_values("Total", ascending=True)

# Plot
plt.figure(figsize=(15, max(3.5, 0.45*len(etd_doc_q))))
y = np.arange(0,len(etd_doc_q.index),1)
left = np.zeros(len(y))

for q in ["Q1","Q2","Q3","Q4","SQ"]:
    vals = etd_doc_q[q].astype(int).values
    plt.barh(y, vals, left=left, label=q)
    left += vals

plt.yticks(y, etd_doc_q.index)
plt.xlabel("Unique publications per researcher (authorship basis)")
plt.title(f"ETD {YEAR_FOCUS} – Publications by researcher and quartile (authorship basis)")
plt.grid(alpha=0.5)
plt.legend(ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.08))

savefig(f"ETD_{YEAR_FOCUS}_researchers_stackedbar_authorship.png")

# =========================
# FIGURE ETD: Pie (authorship basis)
# =========================
ETD_NAME = "ESCUELA DE TRANSFORMACIÓN DIGITAL"   # debe coincidir EXACTO con tu Base

etd_ac = author_contrib[
    (author_contrib["Year"] == YEAR_FOCUS) &
    (author_contrib["ESCUELA"] == ETD_NAME)
].copy()

etd_auth = summary_counts(etd_ac)  # usa tu función summary_counts()

pie_data = etd_auth.loc[["Q1","Q2","Q3","Q4","SQ"], "count"].astype(int).values
pie_labels = ["Q1","Q2","Q3","Q4","SQ"]

plt.figure(figsize=(3,3))
plt.pie(
    pie_data,
    labels=pie_labels,
    autopct=lambda p: f"{p:.1f}%" if p > 0 else ""
)
plt.title(
    f"ETD {YEAR_FOCUS} – Authorship contributions by quartile "
    f"(Total={int(etd_auth.loc['Total','count'])})"
)

savefig(f"ETD_{YEAR_FOCUS}_pie_authorship_quartiles.png")

# =========================
# FIGURE 1: Pie (authorship basis)
# =========================
pie_data = uni_auth.loc[["Q1","Q2","Q3","Q4","SQ"], "count"].astype(int).values
pie_labels = ["Q1","Q2","Q3","Q4","SQ"]

plt.figure(figsize=(3,3))
plt.pie(pie_data, labels=pie_labels,
        autopct=lambda p: f"{p:.1f}%" if p > 0 else "")
plt.title(f"UTB {YEAR_FOCUS} – Authorship contributions by quartile (Total={int(uni_auth.loc['Total','count'])})")
savefig(f"UTB_{YEAR_FOCUS}_pie_authorship_quartiles.png")

# =========================
# FIGURE 2: Stacked bar by school (participation basis)
# =========================
unit_table_sorted = unit_table.sort_values("Total", ascending=True)
plt.figure(figsize=(15, 5))
y = np.arange(len(unit_table_sorted.index))
left = np.zeros(len(y))

for q in ["Q1","Q2","Q3","Q4","SQ"]:
    vals = unit_table_sorted[q].astype(int).values
    plt.barh(y, vals, left=left, label=q)
    left += vals

plt.yticks(y, unit_table_sorted.index)
plt.xlabel("Unique publications (school–paper participation)")
plt.title(f"UTB {YEAR_FOCUS} – Publications by school and quartile (participation basis)")
plt.legend(ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.12))
plt.grid(alpha=0.3)
savefig(f"UTB_{YEAR_FOCUS}_schools_stackedbar_participation.png")

# =========================
# FIGURE 3: Heatmap school × quartile (participation)
# =========================
heat = unit_table_sorted[["Q1","Q2","Q3","Q4","SQ","Total"]].astype(int)

plt.figure(figsize=(15, 4))
im = plt.imshow(heat.values, aspect="auto", cmap="Blues")
plt.xticks(range(heat.shape[1]), heat.columns, rotation=0)
plt.yticks(range(heat.shape[0]), heat.index)
plt.title(f"UTB {YEAR_FOCUS} – Heatmap of counts by school and quartile (participation)")

for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        plt.text(j, i, str(heat.iat[i, j]), ha="center", va="center", fontsize=20)

plt.colorbar(im, fraction=0.046, pad=0.04)
savefig(f"UTB_{YEAR_FOCUS}_heatmap_school_quartile_participation.png")

# =========================
# FIGURE 4: Top 25 researchers (authorship basis, stacked)
# =========================
ac_yr = author_contrib[author_contrib["Year"]==YEAR_FOCUS].copy()
doc_q = (ac_yr.groupby(["DOCENTE","quartile"])["pub_id"].nunique()
         .unstack(fill_value=0)
         .reindex(columns=["Q1","Q2","Q3","Q4","SQ"], fill_value=0))
doc_q["Total"] = doc_q.sum(axis=1)

TOPN = 25
doc_top = doc_q.sort_values("Total", ascending=False).head(TOPN).sort_values("Total", ascending=True)

plt.figure(figsize=(15, 7))
y = np.arange(0,len(doc_top.index),1)
left = np.zeros(len(y))

for q in ["Q1","Q2","Q3","Q4","SQ"]:
    vals = doc_top[q].astype(int).values
    plt.barh(y, vals, left=left, label=q)
    left += vals

plt.yticks(y, doc_top.index)
plt.xlabel("Unique publications per researcher (authorship basis)")
plt.title(f"UTB {YEAR_FOCUS} – Top {TOPN} researchers by publications (authorship basis)")
plt.legend(ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.06))
plt.grid(alpha=0.3)
savefig(f"UTB_{YEAR_FOCUS}_top{TOPN}_researchers_stackedbar_authorship.png")

# =========================
# FIGURE 5: Collaboration matrix between schools (papers linking ≥2 schools)
# =========================
units = sorted(base["ESCUELA"].dropna().unique())
collab = pd.DataFrame(0, index=units, columns=units, dtype=int)
pub_year = pub_utb[pub_utb["Year"]==YEAR_FOCUS]

for _, r in pub_year.iterrows():
    escs = r["escuelas"]
    if not escs or len(escs) < 2:
        continue
    escs = [e for e in escs if e in collab.index]
    for a, b in combinations(sorted(set(escs)), 2):
        collab.loc[a, b] += 1
        collab.loc[b, a] += 1

plt.figure(figsize=(15, 6))
im = plt.imshow(collab.values, aspect="auto", cmap="Blues")
plt.xticks(range(len(units)), units, rotation=20, ha="right")
plt.yticks(range(len(units)), units)
plt.title(f"UTB {YEAR_FOCUS} – Cross-school coauthorship (papers linking schools)")

for i in range(collab.shape[0]):
    for j in range(collab.shape[1]):
        plt.text(j, i, str(int(collab.iat[i, j])), ha="center", va="center", fontsize=20)

plt.colorbar(im, fraction=0.046, pad=0.04)
savefig(f"UTB_{YEAR_FOCUS}_collaboration_matrix_schools.png")


# =========================
# OPTIONAL: zip figures
# =========================
zip_path = os.path.join(OUT_DIR, f"figures_scopus_metrics_{YEAR_FOCUS}.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir(FIG_DIR)):
        if fn.lower().endswith(".png"):
            z.write(os.path.join(FIG_DIR, fn), arcname=fn)

print("OK ✅ Figuras guardadas en:", FIG_DIR)
print("ZIP:", zip_path)

OK ✅ Figuras guardadas en: ./outputs/figures
ZIP: ./outputs/figures_scopus_metrics_2025.zip
